# 네이버 웹툰
이미지 다운로드 + header

In [1]:
# 네이버 웹툰은
# 한 편이 여러개의 이미지로 잘개 쪼개져 있다
# 심지어 이미지 다운로드는 header 에 추가정보가 있어야 제대로 response 된다.

In [2]:
import requests
from bs4 import BeautifulSoup

import os
import re
import urllib.parse as urlparser  # 파일명 추출
from os.path import basename, splitext, split
from IPython.display import Image
from pprint import pprint

# 특정 에피소드 이미지 url 목록 추출

In [3]:
url = "https://comic.naver.com/webtoon/detail?titleId=800770&no=193"

In [4]:
response = requests.get(url)
response

<Response [200]>

In [5]:
soup = BeautifulSoup(response.text, 'html.parser')

## 에피소드 타이틀

In [6]:
title = soup.select_one("#subTitle_toolbar").text.strip()
title

'193화'

## 이미지 목록

In [7]:
img_elements = soup.select("#comic_view_area > #sectionContWide > img")
len(img_elements)

105

In [8]:
img_urls = [
    element.attrs['src']
    for element in img_elements
]

img_urls

['https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_1.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_2.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_3.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_4.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_5.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_6.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_7.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_8.jpg',
 'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544

## 이미지 한개 다운로드

In [9]:
img_url = img_urls[0]
img_url

'https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_1.jpg'

In [10]:
response = requests.get(img_url)
response

<Response [403]>

In [11]:
# 403 에러 발생.

# 브라우저에선 정상 요청 되는데, requests 로는 안된다?

# header 정보 등을 의심해보자.
# user-agent, referer 등...

In [23]:
headers = {
    "Referer": url,
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
}

In [24]:
response = requests.get(img_url, headers=headers) # header 정보 포함.
response

<Response [200]>

In [25]:
out_path = os.path.join('out', 'webtoon')

if not os.path.exists(out_path):
    os.makedirs(out_path)

with open(os.path.join(out_path, 'naverwebtoon.jpg'), 'wb') as f: # 'wb' write binary
    f.write(response.content)

# 특정 작품 (titleId)의 특정 회자(no) 이미지들 크롤링

In [26]:
"4".zfill(4)

'0004'

In [27]:
"320".zfill(4)

'0320'

In [36]:
def download_naver_webtoon(titleId, no, padding=0):
    
    pad_no = f'%0{padding}d' % (no) if padding > 0 else no
    
    # 1. 에피소드 페이지 로딩 -> img url 저장
    url = f'https://comic.naver.com/webtoon/detail.nhn?titleId={titleId}&no={no}'
    response = requests.get(url)

    if response.status_code != 200:
        print('페이지 로딩 실패')
        return
    
    dom = BeautifulSoup(response.text, "html.parser")

    # 에피소드 타이틀    
    title = dom.select_one("#subTitle_toolbar").text.strip()

    # 이미지 url(들)
    img_elements = dom.select("#comic_view_area .wt_viewer img")
    print(no, title, len(img_elements), '개 이미지') # 테스트
    img_urls = [
        element.attrs['src']
        for element in img_elements
    ]

    headers = {
        "Referer": url,
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
    }

    for img_url in img_urls:
        # 파일명 추출
        disassembled = urlparser.urlparse(img_url)
        filename = basename(disassembled.path)

        save_path = os.path.join(out_path, filename)
        print(f"다운로드: {img_url} -> {save_path}")

        response = requests.get(img_url, headers=headers, stream=True)
        if response.status_code != 200:
            print('실패')
            continue
        
        with open(save_path, 'wb') as f:
            f.write(response.content)   
            print('성공')


# 테스트
titleId = 800770
no = 193
download_naver_webtoon(titleId, no)

200
193 193화 105 개 이미지
다운로드: https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_1.jpg -> out/webtoon/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_1.jpg
성공
다운로드: https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_2.jpg -> out/webtoon/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_2.jpg
성공
다운로드: https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_3.jpg -> out/webtoon/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_3.jpg
성공
다운로드: https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_4.jpg -> out/webtoon/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_4.jpg
성공
다운로드: https://image-comic.pstatic.net/webtoon/800770/193/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_5.jpg -> out/webtoon/20260303171817_6544f04ff1b3c192980a3baec6b49b35_IMAG01_5

In [38]:
def download_naver_webtoon(titleId, no, padding=0):
    
    pad_no = f'%0{padding}d' % (no) if padding > 0 else no
    
    # 1. 에피소드 페이지 로딩 -> img url 저장
    url = f'https://comic.naver.com/webtoon/detail?titleId={titleId}&no={no}'
    response = requests.get(url)
    
    print(response.status_code)
    print(response)
    
    if response.status_code != 200:
        print('페이지 로딩 실패')
        return
    
    dom = BeautifulSoup(response.text, "html.parser")

    # 에피소드 타이틀    
    title = dom.select_one("#subTitle_toolbar").text.strip()

    # 이미지 url(들)
    img_elements = dom.select("#comic_view_area .wt_viewer img")
    print(no, title, len(img_elements), '개 이미지') # 테스트
    img_urls = [
        element.attrs['src']
        for element in img_elements
    ]

    headers = {
        "Referer": url,
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36"
    }

    for img_url in img_urls:
        # 파일명 추출
        disassembled = urlparser.urlparse(img_url)
        filename = basename(disassembled.path)

        save_path = os.path.join(out_path, filename)
        print(f"다운로드: {img_url} -> {save_path}")

        response = requests.get(img_url, headers=headers, stream=True)
        if response.status_code != 200:
            print('실패')
            continue
        
        with open(save_path, 'wb') as f:
            f.write(response.content)   
            print('성공')


# 테스트
titleId = 843116
no = 36
download_naver_webtoon(titleId, no)

200
<Response [200]>


AttributeError: 'NoneType' object has no attribute 'text'

In [48]:
import os
import requests
from bs4 import BeautifulSoup
import urllib.parse as urlparser
from os.path import basename

def download_naver_webtoon(titleId, no):
    # 최신 네이버 웹툰 URL 구조
    url = f'https://comic.naver.com/webtoon/detail?titleId={titleId}&no={no}'
    
    # 네이버 차단을 피하기 위해 첫 요청부터 Headers 필수 포함
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://comic.naver.com/webtoon/list?titleId=" + str(titleId)
    }
    
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f'페이지 로딩 실패 (상태코드: {response.status_code})')
        return
    
    dom = BeautifulSoup(response.text, "html.parser")

    # 1. 에피소드 타이틀 추출 (현재 클래스 구조 반영)
    title_element = dom.select_one("#subTitle_toolbar")
    title = title_element.text.strip() if title_element else f"Episode_{no}"

    print(title_element)
    print(title)

    # 2. 이미지 URL 추출 (지연 로딩 대응 및 현재 웹툰 뷰어 클래스 반영)
    # 네이버 웹툰은 보통 특정 클래스 내부의 img 태그들을 사용합니다.
    img_elements = dom.select("div[class*='EpisodeViewer__comic_area'] img")
    
    img_urls = []
    for element in img_elements:
        # data-src 속성이 있으면 지연 로딩용 진짜 이미지이므로 우선 채택, 없으면 src 사용
        img_url = element.get('data-src') or element.get('src')
        if img_url and 'http' in img_url:  # 정상적인 URL만 필터링
            img_urls.append(img_url)

        print(element)
        print(img_url)

    print(f"화수: {no} | 제목: {title} | 이미지 개수: {len(img_urls)}개")

    # 저장할 디렉토리 생성 (예시)
    out_path = f"./webtoon_{titleId}"
    if not os.path.exists(out_path):
        os.makedirs(out_path)

    # 3. 이미지 다운로드 진행
    for idx, img_url in enumerate(img_urls):
        disassembled = urlparser.urlparse(img_url)
        filename = f"{no}_{idx:03d}_{basename(disassembled.path)}" # 정렬을 위해 인덱스 추가

        save_path = os.path.join(out_path, filename)

        # 이미지 요청 시에도 동일한 헤더(특히 Referer)가 있어야 네이버가 이미지를 보내줍니다.
        img_response = requests.get(img_url, headers=headers, stream=True)
        if img_response.status_code != 200:
            print(f'{filename} 다운로드 실패')
            continue
        
        with open(save_path, 'wb') as f:
            f.write(img_response.content)   
            
    print('모든 이미지 다운로드 완료')

# 테스트 (요청하신 최근 웹툰 ID와 화수)
titleId = 843116
no = 36
download_naver_webtoon(titleId, no)

None
Episode_36
화수: 36 | 제목: Episode_36 | 이미지 개수: 0개
모든 이미지 다운로드 완료
